Импорт библиотек

In [1]:
import pandas as pd
import requests
import os
import time
from dotenv import load_dotenv
from tqdm import tqdm

Настройка API-ключа и серверов

In [2]:
load_dotenv()

api_key = os.getenv("api_key") #Ключ из .env
regions = ["ru", "euw1", "na1"] #Сервер
tiers = ["challenger", "grandmaster", "master"] #Ранг
queue = "RANKED_SOLO_5x5" 
delay = 0.85 #Задержка

headers = {
    "X-Riot-Token": api_key 
}

Получаем список топовых игроков Challenger, Grandmaster, Master для регионов euw1 и na1, ru

In [3]:
all_entries = []

for region in regions:
  for tier in tiers:
    url = f"https://{region}.api.riotgames.com/lol/league/v4/{tier}leagues/by-queue/{queue}?api_key={api_key}"

    try:
      response = requests.get(url, timeout= 20, headers= headers)
      response.raise_for_status()
      data = response.json()

      entries = pd.DataFrame(data["entries"])
      entries['region'] = region.upper()
      entries['tier'] = tier.upper()

      all_entries.append(entries)
      time.sleep(delay)

    except requests.exceptions.RequestException as e:
        print(f"Ошибка для {region}/{tier}: {e}")

df_entries = pd.concat(all_entries, ignore_index=True)
df_entries = df_entries[["puuid", "leaguePoints", "wins", "losses", "tier", "region"]].rename(columns= {'puuid': 'puu_id',
                 'leaguePoints': 'league_points',
                 'tier': 'range',
                 "region": "server"})

Информация о summoner по PUUID

In [4]:
win_500 = df_entries[df_entries['wins'] > 500].copy().reset_index()
players_data = win_500.copy()

players_data['region'] = players_data['server'].apply(lambda x: "americas" if x == "NA1" else "europe")
players_data = players_data[["puu_id", "region", "server", "range", "league_points", "wins", "losses"]]
players_data['winrate_%'] = round((players_data['wins'] / (players_data['wins'] + players_data['losses'])) * 100, 2)
players_data['matches_month'] = players_data['wins'] + players_data['losses']

players_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   puu_id         700 non-null    object 
 1   region         700 non-null    object 
 2   server         700 non-null    object 
 3   range          700 non-null    object 
 4   league_points  700 non-null    int64  
 5   wins           700 non-null    int64  
 6   losses         700 non-null    int64  
 7   winrate_%      700 non-null    float64
 8   matches_month  700 non-null    int64  
dtypes: float64(1), int64(4), object(4)
memory usage: 49.3+ KB


In [5]:
# Сохраняем таблицу матчей в CSV
players_data.to_csv("players.csv", index=False)
print("Сохранено: players.csv")

Сохранено: players.csv


Получаем матчи игроков

In [6]:
all_matches_id = []

for region, group in players_data.groupby("region"):
  for puuid in group['puu_id']:
    url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?queue=420&type=ranked&count=3&api_key={api_key}"

    try:
      response = requests.get(url, timeout= 20, headers= headers)
      response.raise_for_status()
      data = response.json()

      all_matches_id.extend(data)
      # Добавляем задержку после каждого запроса
      time.sleep(delay)

    except requests.exceptions.RequestException as e:
      print(f"Ошибка для {puuid} в регионе {region}: {e}")

# Удаляем дубликаты
match_ids = list(set(all_matches_id))

print("Количество матчей:", len(all_matches_id))
print(all_matches_id[:5])

df_matches_id = pd.DataFrame(all_matches_id)

Ошибка для JcuXv8NiHA9yW4P-y7LBDvFn84bx4xoLt_EPZEqbudVZfWOa2WcoDcptAVq-sy4g5hKdiBSkWLuIGA в регионе americas: HTTPSConnectionPool(host='americas.api.riotgames.com', port=443): Max retries exceeded with url: /lol/match/v5/matches/by-puuid/JcuXv8NiHA9yW4P-y7LBDvFn84bx4xoLt_EPZEqbudVZfWOa2WcoDcptAVq-sy4g5hKdiBSkWLuIGA/ids?queue=420&type=ranked&count=3&api_key=RGAPI-b1f123d1-e687-4c5e-85ab-601137764d82 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x00000207942FAC10>, 'Connection to americas.api.riotgames.com timed out. (connect timeout=20)'))
Ошибка для Dy5IIMJGA6nWe3ScZqyfuCwUak54akdsT3YUl13hmPCrj_60WC7CeXUKCGqwVw1zgPRXhKWqwqHtAQ в регионе americas: HTTPSConnectionPool(host='americas.api.riotgames.com', port=443): Max retries exceeded with url: /lol/match/v5/matches/by-puuid/Dy5IIMJGA6nWe3ScZqyfuCwUak54akdsT3YUl13hmPCrj_60WC7CeXUKCGqwVw1zgPRXhKWqwqHtAQ/ids?queue=420&type=ranked&count=3&api_key=RGAPI-b1f123d1-e687-4c5e-85ab-601137764d82 (Caused by ConnectTim

In [7]:
# Предварительно создаём список кортежей (region, match_id)
df_matches_id['region'] = df_matches_id[0].apply(lambda x: 'americas' if x.startswith('NA1_') else 'europe')

tasks = []

for region, group in df_matches_id.groupby("region"):
    for match_id in group[0]:  
        tasks.append((region, match_id))

In [8]:
all_matches = []

for region, match_id in tqdm(tasks, desc="Загрузка матчей", unit="матч"):
    url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/{match_id}"

    try:
        response = requests.get(url, timeout= 20, headers=headers)
        response.raise_for_status()
        data = response.json()
        time.sleep(delay)
    except Exception as e:
        tqdm.write(f"Пропускаем {match_id}: {e}")   # пишет поверх прогресс-бара
        continue

    # В ответе два раздела: 'metadata' и 'info'
    # Нам нужен 'info' — там вся игровая информация
    game_info = data["info"]
    game_duration = game_info["gameDuration"]  # длительность в секундах
    participants = game_info["participants"]   # список 10 игроков

    # извлекаем нужные поля по каждому участнику
    for p in participants:

      row = {
              "match_id":      match_id,
              "puuid":         p.get("puuid"),
              "champion":      p.get("championName"),
              "kills":         p.get("kills"),
              "deaths":        p.get("deaths"),
              "assists":       p.get("assists"),
              "gold_earned":   p.get("goldEarned"),
              "damage_to_champions": p.get("totalDamageDealtToChampions"),
              "minions_killed":      p.get("totalMinionsKilled"),
              "vision_score":        p.get("visionScore"),
              "win":           p.get("win"),        # True/False для конкретного игрока
              "team_position": p.get("teamPosition"),
              "game_duration_sec": game_duration,
              "game_version": data["info"]["gameVersion"],

              # Предметы
              "item0": p.get("item0"),
              "item1": p.get("item1"),
              "item2": p.get("item2"),
              "item3": p.get("item3"),
              "item4": p.get("item4"),
              "item5": p.get("item5"),
              }
      all_matches.append(row)



print(f"\nВсего собрано строк: {len(all_matches)}")

Загрузка матчей:   5%|▌         | 107/2085 [05:22<4:15:08,  7.74s/матч]

Пропускаем NA1_5576211319: HTTPSConnectionPool(host='americas.api.riotgames.com', port=443): Max retries exceeded with url: /lol/match/v5/matches/NA1_5576211319 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000207942FAC10>: Failed to resolve 'americas.api.riotgames.com' ([Errno 11001] getaddrinfo failed)"))


Загрузка матчей:   6%|▌         | 116/2085 [05:59<4:41:18,  8.57s/матч]

Пропускаем NA1_5576350096: Expecting value: line 1 column 1 (char 0)


Загрузка матчей:  16%|█▌        | 332/2085 [16:45<6:28:11, 13.29s/матч]

...

Пропускаем EUW1_7879126039: HTTPSConnectionPool(host='europe.api.riotgames.com', port=443): Max retries exceeded with url: /lol/match/v5/matches/EUW1_7879126039 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x00000207942FAC10>, 'Connection to europe.api.riotgames.com timed out. (connect timeout=20)'))


Загрузка матчей:  96%|█████████▋| 2011/2085 [1:38:02<10:38,  8.62s/матч]

Пропускаем EUW1_7874934348: HTTPSConnectionPool(host='europe.api.riotgames.com', port=443): Max retries exceeded with url: /lol/match/v5/matches/EUW1_7874934348 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000207942FB390>: Failed to resolve 'europe.api.riotgames.com' ([Errno 11001] getaddrinfo failed)"))


Загрузка матчей: 100%|██████████| 2085/2085 [1:40:55<00:00,  2.90s/матч]


Всего собрано строк: 20740


In [9]:
# Создаём DataFrame из всех собранных строк
matches_data = pd.DataFrame(all_matches)

matches_data.head()

,match_id,puuid,champion,kills,deaths,assists,gold_earned,damage_to_champions,minions_killed,vision_score,win,team_position,game_duration_sec,game_version,item0,item1,item2,item3,item4,item5
0,NA1_5576615831,3ES__2h8z78Zwn5yRx3B-M_fOOuZI-6J41oDTgi4DSMYIK...,Rumble,6,8,3,12907,30526,253,52,False,TOP,2006,16.11.782.9736,3152,6653,3157,3145,3047,0
1,NA1_5576615831,tUfQUcX2PNY5X3S07_qcc1CFrfFBIPzDV-Of35l6bqzh3M...,MasterYi,10,4,0,17801,21118,81,32,False,JUNGLE,2006,16.11.782.9736,3153,6672,3047,3156,6333,1038
2,NA1_5576615831,CnRl8ZEh8ui_ZSUJ-vj48SROqekGpOaf0vD32w9rW4OWwr...,Veigar,1,10,1,10982,6568,252,31,False,MIDDLE,2006,16.11.782.9736,1058,3152,3040,3170,3066,3113
3,NA1_5576615831,O1fcMdHW6RJhaxQNCgOp2B1GjD57xK5sy1dgnCohQIPjte...,Ezreal,3,8,4,15137,24640,283,17,False,BOTTOM,2006,16.11.782.9736,6694,3042,3078,2517,3082,1029
4,NA1_5576615831,sU14RD1R1BAaQHsveJmWuELTU2ca8sEjPOixxvNuovoieC...,Janna,2,7,12,9686,8754,29,127,False,UTILITY,2006,16.11.782.9736,3222,3870,3158,4005,4642,3113


In [11]:
# Смотрим типы данных и пропуски
matches_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20740 entries, 0 to 20739
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   match_id             20740 non-null  object
 1   puuid                20740 non-null  object
 2   champion             20740 non-null  object
 3   kills                20740 non-null  int64 
 4   deaths               20740 non-null  int64 
 5   assists              20740 non-null  int64 
 6   gold_earned          20740 non-null  int64 
 7   damage_to_champions  20740 non-null  int64 
 8   minions_killed       20740 non-null  int64 
 9   vision_score         20740 non-null  int64 
 10  win                  20740 non-null  bool  
 11  team_position        20740 non-null  object
 12  game_duration_sec    20740 non-null  int64 
 13  game_version         20740 non-null  object
 14  item0                20740 non-null  int64 
 15  item1                20740 non-null  int64 
 16  item

In [12]:
# Сохраняем таблицу матчей в CSV
matches_data.to_csv("matches.csv", index=False)
print("Сохранено: matches.csv")

Сохранено: matches.csv


Получаем справочник чемпионов и предметов

In [13]:
version_url = "https://ddragon.leagueoflegends.com/api/versions.json"

versions = requests.get(version_url).json()

latest_version = versions[0]

print("Последняя версия:", latest_version)

Последняя версия: 16.11.1


In [15]:
url = f"https://ddragon.leagueoflegends.com/cdn/{latest_version}/data/en_US/champion.json"

response = requests.get(url)
response.raise_for_status()
data = response.json()

champions_info = []

for name, info in data['data'].items():
  champions = {"champion": name,
               "tags": ",".join(info['tags'])
  }
  champions_info.append(champions)

champions_data = pd.DataFrame(champions_info)

print("Количество героев:", len(champions_data))
champions_data.head(3)

Количество героев: 172


,champion,tags
0,Aatrox,Fighter
1,Ahri,"Mage,Assassin"
2,Akali,Assassin


In [16]:
# Сохраняем справочник чемпионов
champions_data.to_csv("champions.csv", index=False)
print("Сохранено: champions.csv")

Сохранено: champions.csv


In [17]:
url = f"https://ddragon.leagueoflegends.com/cdn/{latest_version}/data/en_US/item.json"

response = requests.get(url)
response.raise_for_status()
data = response.json()

items_info = []

for item_id, item_info in data["data"].items():

    row = {
        "item_id": item_id,
        "item_name": item_info.get("name"),
        "gold_total": item_info.get("gold", {}).get("total"),
        "gold_sell": item_info.get("gold", {}).get("sell"),
    }

    items_info.append(row)

items_data = pd.DataFrame(items_info)

print("Количество предметов:", len(items_data))
items_data.head()

Количество предметов: 705


,item_id,item_name,gold_total,gold_sell
0,1001,Boots,300,210
1,1004,Faerie Charm,200,140
2,1006,Rejuvenation Bead,300,120
3,1011,Giant's Belt,900,630
4,1018,Cloak of Agility,600,420


In [18]:
# Сохраняем справочник предметов
items_data.to_csv("items.csv", index=False)
print("Сохранено: items.csv")

Сохранено: items.csv


In [19]:
os.makedirs("parquet_data", exist_ok=True)

# Сохраняем таблицы
players_data.to_parquet(
    "parquet_data/players.parquet",
    index=False, engine='fastparquet'
)

matches_data.to_parquet(
    "parquet_data/matches.parquet",
    index=False, engine='fastparquet'
)

champions_data.to_parquet(
    "parquet_data/champions.parquet",
    index=False, engine='fastparquet'
)

items_data.to_parquet(
    "parquet_data/items.parquet",
    index=False, engine='fastparquet'
)

print("Parquet файлы успешно созданы")

Parquet файлы успешно созданы
